## Finanzas PUNTO 1 -- actualización y descuento

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('datos/datos_limpio_automovil.xlsx')

In [4]:
df

,Unnamed: 0,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,...,Estudios_Escolar,Estudios_Grado Universitario,Estudios_Máster,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero
0,0,O77MPJ,26,19153,6973,706,9,1,19.05,36,...,False,True,False,False,False,True,False,False,False,True
1,8,IYOL1S,29,39531,40000,826,0,1,9.74,24,...,False,False,False,False,False,True,False,True,False,False
2,9,FRMZVN,29,40348,16448,740,9,1,14.73,24,...,False,False,True,False,False,True,False,False,False,True
3,26,YKHXVY,38,41732,42040,661,173,2,13.24,48,...,False,True,False,False,False,False,True,True,False,False
4,32,NHDJTK,35,35085,16406,759,181,1,9.74,36,...,True,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76919,510685,9DP3XV,36,45006,40252,746,144,1,12.74,36,...,False,True,False,False,False,True,False,True,False,False
76920,510686,R30LLT,42,44124,15509,574,246,2,8.00,24,...,False,True,False,True,False,False,False,False,False,True
76921,510687,W2Y3KE,44,84570,331820,791,246,2,6.90,60,...,False,False,False,False,False,False,True,True,False,False
76922,510695,WVJ9ZC,42,61642,19371,502,223,3,8.55,12,...,False,False,True,False,False,False,True,True,False,False


In [8]:
df.columns

Index(['Unnamed: 0', 'ID', 'Edad', 'Ingresos', 'Monto_Inicial',
       'Scoring_Crediticio', 'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes',
       'Duracion', 'Ratio_Deuda_Ingresos', 'Posesion_Hipoteca',
       'Personas_Cargo', 'Fiador', 'Impago', 'Prima', 'Estudios_Doctorado',
       'Estudios_Escolar', 'Estudios_Grado Universitario', 'Estudios_Máster',
       'Tipo_Jornada_Laboral_Autónomo', 'Tipo_Jornada_Laboral_Desempleado',
       'Tipo_Jornada_Laboral_Jornada completa',
       'Tipo_Jornada_Laboral_Tiempo parcial', 'Estado_Civil_Casado',
       'Estado_Civil_Divorciado', 'Estado_Civil_Soltero'],
      dtype='str')

In [ ]:
df = df[['ID', 'Monto_Inicial', 'Ratio_Interes', 'Duracion']]

Me quedo con las columnas:
- ID: para saber que cliente es
- Monto inicial: Cantidad de dinero acordada.
- Ratio_Interes: El ratio de interés del préstamo (multiplicado por 100).
- Duracion: La duración del prestamo en meses.

Ya que son las variables que vamos a utilizar en las fórmulas.

In [10]:
df

,ID,Monto_Inicial,Ratio_Interes,Duracion
0,O77MPJ,6973,19.05,36
1,IYOL1S,40000,9.74,24
2,FRMZVN,16448,14.73,24
3,YKHXVY,42040,13.24,48
4,NHDJTK,16406,9.74,36
...,...,...,...,...
76919,9DP3XV,40252,12.74,36
76920,R30LLT,15509,8.00,24
76921,W2Y3KE,331820,6.90,60
76922,WVJ9ZC,19371,8.55,12


In [ ]:
# Tipo de interés en tanto por uno
df["i"] = df["Ratio_Interes"] / 100

# Duración expresada en años
df["n"] = df["Duracion"] / 12

# Capital inicial
df["C0_inicial"] = df["Monto_Inicial"]

# ---------------------------------------------------------
# 3. Calcular el capital futuro
# ---------------------------------------------------------
# Capitalización compuesta:
#
# Cn = C0 * (1+i)^n
#
# Esto nos permite obtener el capital que tendría
# cada cliente al finalizar el periodo.

df["Cn"] = df["C0_inicial"] * (1 + df["i"]) ** df["n"]


# ---------------------------------------------------------
# 4. DESCUENTO RACIONAL SIMPLE
# ---------------------------------------------------------
#
# C0 = Cn / (1 + n*i)

df["C0_racional_simple"] = (
    df["Cn"] / (1 + df["n"] * df["i"])
)


# ---------------------------------------------------------
# 5. DESCUENTO RACIONAL COMPUESTO
# ---------------------------------------------------------
#
# C0 = Cn / (1+i)^n

df["C0_racional_compuesto"] = (
    df["Cn"] / (1 + df["i"]) ** df["n"]
)


# ---------------------------------------------------------
# 6. DESCUENTO COMERCIAL SIMPLE
# ---------------------------------------------------------
#
# C0 = Cn * (1 - n*d)
#
# En este ejercicio utilizamos como tasa de descuento
# el mismo tipo porcentual disponible en Ratio_Interes.

df["d"] = df["Ratio_Interes"] / 100

df["C0_comercial_simple"] = (
    df["Cn"] * (1 - df["n"] * df["d"])
)


# ---------------------------------------------------------
# 7. DESCUENTO COMERCIAL COMPUESTO
# ---------------------------------------------------------
#
# C0 = Cn * (1-d)^n

df["C0_comercial_compuesto"] = (
    df["Cn"] * (1 - df["d"]) ** df["n"]
)


# ---------------------------------------------------------
# 8. Calcular el descuento en euros
# ---------------------------------------------------------

df["D_racional_simple"] = (
    df["Cn"] - df["C0_racional_simple"]
)

df["D_racional_compuesto"] = (
    df["Cn"] - df["C0_racional_compuesto"]
)

df["D_comercial_simple"] = (
    df["Cn"] - df["C0_comercial_simple"]
)

df["D_comercial_compuesto"] = (
    df["Cn"] - df["C0_comercial_compuesto"]
)


# ---------------------------------------------------------
# 9. Mostrar los resultados
# ---------------------------------------------------------

columnas_resultados = [
    "ID",
    "Monto_Inicial",
    "Ratio_Interes",
    "Duracion",
    "Cn",
    "C0_racional_simple",
    "C0_racional_compuesto",
    "C0_comercial_simple",
    "C0_comercial_compuesto"
]

print(df[columnas_resultados].head(10))